# Text-to-SQL & semantic parsing — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def softmax(x, axis=-1):
    x=np.asarray(x, dtype=float); x=x-x.max(axis=axis, keepdims=True); e=np.exp(x); return e/e.sum(axis=axis, keepdims=True)
def sigmoid(x): return 1/(1+np.exp(-np.asarray(x, dtype=float)))
def normalize(v):
    v=np.asarray(v, dtype=float); n=np.linalg.norm(v); return v/(n if n else 1)
def edit_distance(a,b):
    D=np.zeros((len(a)+1,len(b)+1), dtype=int); D[:,0]=np.arange(len(a)+1); D[0,:]=np.arange(len(b)+1)
    for i in range(1,len(a)+1):
        for j in range(1,len(b)+1):
            D[i,j]=min(D[i-1,j]+1, D[i,j-1]+1, D[i-1,j-1]+(a[i-1]!=b[j-1]))
    return D
print('setup ok')

## Map natural language to executable meaning under schema constraints
The examples score schema links, sketch actions, enforce grammar masks, execute denotations and compare exact match with execution accuracy.

In [ ]:
words=['highest','sales']; cols=['revenue','region']; link=np.array([[.8,.1],[.9,.2]])
plt.figure(figsize=(4,3)); plt.imshow(link,cmap='Blues'); plt.xticks(range(2),cols); plt.yticks(range(2),words); plt.title('Question words link to schema columns')
assert link[1,0]>link[1,1]

In [ ]:
actions=['SELECT','AGG_MAX','COLUMN_revenue','FROM_sales']; steps=np.arange(len(actions))
plt.figure(figsize=(6,2)); plt.plot(steps,steps,'-o'); plt.xticks(steps,actions,rotation=30); plt.title('Parser emits a constrained action sequence')
assert actions[0]=='SELECT' and actions[1]=='AGG_MAX'

In [ ]:
valid=np.array([1,0,1,0]); logits=np.array([2,5,1,4.],float); masked=np.where(valid,logits,-1e9); p=softmax(masked)
plt.figure(figsize=(5,3)); plt.bar(range(4),p); plt.title('Grammar mask removes invalid next tokens')
assert p[1]==0 and p[3]==0

In [ ]:
rows=np.array([10,30,20]); ans=rows.max()
plt.figure(figsize=(5,3)); plt.bar(['r0','r1','r2'],rows); plt.axhline(ans,ls='--',c='r'); plt.title('SQL execution grounds meaning in denotation')
assert ans==30

In [ ]:
exact=np.array([1,0,0]); exec_acc=np.array([1,1,0]); plt.figure(figsize=(5,3)); x=np.arange(3); plt.bar(x-.15,exact,.3,label='exact'); plt.bar(x+.15,exec_acc,.3,label='execution'); plt.legend(); plt.title('Different SQL strings can execute to same answer')
assert exec_acc.sum()>exact.sum()